# Validadores customizados — introdução (slides da aula)

Os slides originais (em inglês) estão abaixo, cada um seguido do resumo em português.

## 1. Qual é o valor de entrada do validador?

![slide 1](assets/slide_01.png)

Um validador customizado pode receber o valor em dois momentos:

- **before** (antes): o valor **cru**, exatamente como veio nos dados desserializados;
- **after** (depois): o valor já **convertido e validado** pelo Pydantic segundo o *type hint* do campo.

```
                     ┌──────────────────────┐
  dados ──────────►  │   TYPE HINT DO CAMPO │ ──────────►  campo da instância
                     │  (validação Pydantic)│
                     └──────────────────────┘
          ▲ dados crus                         ▲ dados já validados
   [validador BEFORE]                    [validador AFTER]
```

## 2. Vários validadores no mesmo campo

![slide 2](assets/slide_02.png)

Um campo pode ter vários validadores customizados, misturando before e after:

```
dados ─► [before 1] ─► [before 2] ─► [before 3] ─► (validação Pydantic) ─► [after 4] ─► [after 5] ─► [after 6] ─► valor final
```

## 3 e 4. Ordem de execução

![slide 3](assets/slide_03.png)

![slide 4](assets/slide_04.png)

A ordem segue a ordem das funções na classe, mas com sentidos opostos:

- validadores **before** rodam de **baixo para cima** (o último definido roda primeiro);
- validadores **after** rodam de **cima para baixo** (o primeiro definido roda primeiro).

Mesmo intercalando before e after na classe, o Pydantic varre os before de baixo para cima, roda a validação do tipo e depois varre os
after de cima para baixo:

| ordem na classe | ordem de execução |
|---|---|
| `before_1`, `after_1`, `before_2`, `after_2`, `before_3`, `after_3` | `before_3` → `before_2` → `before_1` → **Pydantic** → `after_1` → `after_2` → `after_3` |

## 5. Como um validador indica falha

![slide 5](assets/slide_05.png)

- Levante **`ValueError`** com uma boa descrição — o Pydantic a converte em `ValidationError`.
- A documentação também permite:
  - **`assert`** — **não use**: o Python pode rodar com `-O`, que **desliga todos os asserts** (a validação simplesmente some);
  - **`PydanticCustomError`** — raramente necessário, mas mais flexível para reportar o erro (código e contexto próprios).
- Essas exceções viram `ValidationError`.
- **Qualquer outra exceção** (`TypeError`, `KeyError`...) **sobe como está**, sem virar `ValidationError`.

As próximas aulas demonstram cada um desses pontos com código.